In [2]:
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 87.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 2. Configuration & Model Loading
repo_id = "chanystrange/mistral-agri-merged_143"
torch.cuda.empty_cache()

# 4-bit quantization to fit on T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(repo_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

# 3. Initialize the Generator
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("\n✅ Model and Pipeline Loaded Successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


✅ Model and Pipeline Loaded Successfully!


In [ ]:
import torch

print("="*50)
print("--- Agri-AI Advisor (Type 'exit' to quit) ---")
print("="*50)

while True:
    user_query = input("\n🌾 Ask your agriculture question: ")

    if user_query.lower() in ['exit', 'quit', 'stop']:
        print("Goodbye!")
        break

    # Visual feedback while the GPU processes
    print("\n⏳ Thinking... please wait.")

    # Mistral Instruction format
    prompt = f"<s>[INST] {user_query} [/INST]"

    with torch.no_grad():
        results = generator(
            prompt,
            max_new_tokens=250,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            generation_config=None  # Clears conflicting default warnings
        )

    # Extract only the response part
    answer = results[0]['generated_text'].split("[/INST]")[-1].strip()

    print(f"\n✅ AI Response:\n{answer}\n")
    print("-" * 50)

--- Agri-AI Advisor (Type 'exit' to quit) ---

🌾 Ask your agriculture question: i want to cultivate paddy

⏳ Thinking... please wait.

✅ AI Response:
Cultivating paddy (Rice) involves several steps:

1. Land Preparation: Choose a well-drained, low-lying area for rice cultivation. Prepare the land by clearing it of weeds and other debris, and tilling it to a depth of about 10-15 cm.

2. Seed Selection: Use high-quality rice seeds suitable for your local conditions.

3. Transplanting: Rice seeds are usually planted using a method called transplanting. Seeds are first sown in nurseries, and when they sprout into seedlings with 2-4 visible leaves, they are transplanted into the field.

4. Field Preparation: Before transplanting, the field is flooded with water to create a suitable environment for rice growth. A shallow layer of soil is mounded on the bottom of the field to support the transplanted seedlings.

5. Cultivation: After transplanting, the field is maintained by weeding, applying